In [169]:
from pymongo import MongoClient
import re
import os
from typing import List, Dict, Any
import shutil
import json

In [170]:
def connect_mongodb(uri,db_name, collection_name):
    # Connect to MongoDB 
    client = MongoClient(uri)
    
    # Select the database and collection
    db = client[db_name]
    collection = db[collection_name]
    
    return collection
def get_all_ids(collection):
    return [doc['_id'] for doc in collection.find({}, {'_id': 1})]
    
def get_document_by_id(collection, target_id):
    return collection.find_one({"_id": target_id})
    
def get_level_3_path(data):
    level_2_path = data.get('level_2_path', [])
    level_3_values = []
    for item in level_2_path:
        level_3 = item.get('level_3_path', [])
        level_3_values.extend(level_3)
    return level_3_values

def get_files_with_prefix_and_extension(directory_path, prefix, extension, recursive=False):
    files = []

    if recursive:
        for root, dirs, filenames in os.walk(directory_path):
            for filename in filenames:
                if filename.startswith(prefix) and filename.endswith(extension):
                    files.append(os.path.join(root, filename))
    else:
        for filename in os.listdir(directory_path):
            if filename.startswith(prefix) and filename.endswith(extension):
                file_path = os.path.join(directory_path, filename)
                if os.path.isfile(file_path):
                    files.append(file_path)

    return files
def get_all_directories(root_path):
    """
    Get all directory names under the given root path.
    Returns a list of directory names.
    """
    return [name for name in os.listdir(root_path) if os.path.isdir(os.path.join(root_path, name))]
def is_directory_exist(dir_path):
    """
    Check if a directory exists.
    Return True if it exists, False otherwise.
    """
    return os.path.exists(dir_path) and os.path.isdir(dir_path)
def get_file_name(file_path):
    return os.path.basename(file_path)
def create_dict(package_name, add_count, remove_count):
    result = {
        package_name: {
            "add": add_count,
            "remove": remove_count
        }
    }
    return result
def sum_add_remove(data):
    total_add = 0
    total_remove = 0

    for item in data:
        for app_data in item.values():  # lấy dict bên trong
            total_add += app_data.get("add", 0)
            total_remove += app_data.get("remove", 0)

    return total_add, total_remove

In [171]:
mongoDB_uri = 'mongodb://localhost:27017'
mongoDB_database = 'wearable-project-2' 
mongoDB_collection = 'codeblock_ast'

In [172]:
# Connect to the MongoDB collection
collection = connect_mongodb(mongoDB_uri,mongoDB_database,mongoDB_collection)

In [173]:
array_package_name = get_all_ids(collection)
print(len(array_package_name))

469


In [174]:
final_arr = []
for x in range(len(array_package_name)):
# for x in range(0,2):
    print("------------ Loop-"+str(x)+" ------------")
    package_name = array_package_name[x]
    print("package name: ",package_name)
    data = get_document_by_id(collection, package_name)
    level_3_directory_array = get_level_3_path(data)
    # add_code_block_arr = []
    # remove_code_block_arr = []
    # add_count = 0
    # remove_count = 0
    add_count_arr = []
    remove_count_arr = []
    for y in range(len(level_3_directory_array)):
        level_3_directory_path = level_3_directory_array[y]
        print("**********level_3_directory_path: ",level_3_directory_path)
        json_add_directory_path = level_3_directory_path+"\\json"
        json_remove_directory_path = level_3_directory_path+"\\json-remove"
        check_add_directory_result = is_directory_exist(json_add_directory_path)
        check_remove_directory_result = is_directory_exist(json_remove_directory_path)
        add_code_block_arr = []
        remove_code_block_arr = []
        if(check_add_directory_result):
            add_code_block = get_files_with_prefix_and_extension(json_add_directory_path, "codeblock-", ".java")
            for z in range(len(add_code_block)):
                file_name = get_file_name(add_code_block[z])
                add_code_block_arr.append(file_name)
        if(check_remove_directory_result):
            remove_code_block = get_files_with_prefix_and_extension(json_remove_directory_path, "codeblock-", ".java")
            for t in range(len(remove_code_block)):
                file_name = get_file_name(remove_code_block[t])
                remove_code_block_arr.append(file_name)
        add_count_arr.append(len(add_code_block_arr))
        remove_count_arr.append(len(remove_code_block_arr))
    print("add_count_arr: ",add_count_arr)
    print("remove_count_arr: ",remove_count_arr)
    result_dict = create_dict(package_name, sum(add_count_arr), sum(remove_count_arr))
    # print(result_dict)
    final_arr.append(result_dict)
    # break

------------ Loop-0 ------------
package name:  adarshurs.android.vlcmobileremote
**********level_3_directory_path:  E:\wearable-2-dataset-code-extraction\adarshurs.android.vlcmobileremote\Google\com.google.android.gms.ads.MobileAds
**********level_3_directory_path:  E:\wearable-2-dataset-code-extraction\adarshurs.android.vlcmobileremote\Google\com.google.firebase.analytics.connector.AnalyticsConnector
**********level_3_directory_path:  E:\wearable-2-dataset-code-extraction\adarshurs.android.vlcmobileremote\Google\com.google.firebase.components.ComponentRegistrar
**********level_3_directory_path:  E:\wearable-2-dataset-code-extraction\adarshurs.android.vlcmobileremote\Google\com.google.firebase.provider.FirebaseInitProvider
add_count_arr:  [3, 34, 21, 7]
remove_count_arr:  [1, 3, 6, 5]
------------ Loop-1 ------------
package name:  app.EnForma.fitcoapp.net
**********level_3_directory_path:  E:\wearable-2-dataset-code-extraction\app.EnForma.fitcoapp.net\original\Firebase
**********leve

In [175]:
print(final_arr)

[{'adarshurs.android.vlcmobileremote': {'add': 65, 'remove': 15}}, {'app.EnForma.fitcoapp.net': {'add': 2933, 'remove': 354}}, {'app.f4square.fitness': {'add': 893, 'remove': 95}}, {'app.fit.meet': {'add': 3081, 'remove': 493}}, {'app.groupcal.www': {'add': 2589, 'remove': 289}}, {'app.growwithjo.diet.fitness': {'add': 496, 'remove': 48}}, {'app.LomasGym.fitcoapp.net': {'add': 15907, 'remove': 2440}}, {'app.mfit.personal': {'add': 296, 'remove': 37}}, {'app.MOVE.fitcoapp.net': {'add': 2932, 'remove': 354}}, {'appnativo.navysportcenter.fitcoapp.net': {'add': 2931, 'remove': 354}}, {'apps.r.compass': {'add': 3208, 'remove': 423}}, {'blacknote.amazfitmaster': {'add': 25, 'remove': 4}}, {'br.com.fitastic.appaluno': {'add': 3337, 'remove': 545}}, {'br.com.mufitness.station.aluno': {'add': 0, 'remove': 0}}, {'br.com.personalfitapp': {'add': 172, 'remove': 30}}, {'br.com.w12.bluefit': {'add': 393, 'remove': 61}}, {'br.com.w12.heyfit': {'add': 131, 'remove': 18}}, {'br.com.w12.redfit': {'add':

In [176]:
total_add, total_remove = sum_add_remove(final_arr)
print(f"Number of add: {total_add}")
print(f"Number of remove: {total_remove}")

Number of add: 551037
Number of remove: 75681
